
## ⚙️ 1. Environment Setup & Imports

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import cv2
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel, mutual_info_classif
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
# from imblearn.over_sampling import SMOTE

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("All imports OK ✓")

---
## 📂 2. Data Loading

In [ ]:
DATA_DIR = "/kaggle/input/competitions/multimodal-skin-classification"
# DATA_DIR = "D:\ML_notebook\multimodal-skin-classification"

train_images  = np.load(os.path.join(DATA_DIR, "train_images.npy"))
train_labels  = np.load(os.path.join(DATA_DIR, "train_labels.npy"))
train_tabular = np.load(os.path.join(DATA_DIR, "train_tabular.npy"))
train_ids     = np.load(os.path.join(DATA_DIR, "train_ids.npy"),   allow_pickle=True)

test_images   = np.load(os.path.join(DATA_DIR, "test_images.npy"))
test_tabular  = np.load(os.path.join(DATA_DIR, "test_tabular.npy"))
test_ids      = np.load(os.path.join(DATA_DIR, "test_ids.npy"),    allow_pickle=True)

metadata = np.load(os.path.join(DATA_DIR, "metadata.npy"), allow_pickle=True).item()
idx_to_class = {v: k for k, v in metadata['class_to_idx'].items()}

print(f"Train images  : {train_images.shape}  dtype={train_images.dtype}")
print(f"Train labels  : {train_labels.shape}")
print(f"Train tabular : {train_tabular.shape}  (age, sex, anatom_site)")
print(f"Test  images  : {test_images.shape}")
print(f"Class names   : {metadata['class_names']}")


---
## **3. Hair removal**

In [ ]:
import cv2
import numpy as np

def remove_hair_adaptive(image, threshold_sensitivity=18):
    # 1. Grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    
    # 2. Blackhat with a slightly smaller kernel to target thin hairs only
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    
    # 3. Create mask with higher threshold to avoid lesion textures
    _, mask = cv2.threshold(blackhat, threshold_sensitivity, 255, cv2.THRESH_BINARY)
    
    # --- DETECTION LOGIC ---
    # Calculate what percentage of the image is "hair"
    hair_density = np.sum(mask > 0) / (mask.shape[0] * mask.shape[1])
    
    # If density is extremely low (e.g., < 0.5%) or zero, return original image
    # This prevents the filter from smoothing "clean" lesion images
    if hair_density < 0.005: 
        return image
    
    # 4. Inpaint only if hair is detected
    # Using a smaller inpaintRadius to preserve lesion borders
    dst = cv2.inpaint(image, mask, inpaintRadius=1, flags=cv2.INPAINT_TELEA)
    return dst

def process_batch_adaptive(imgs):
    out = np.empty_like(imgs)
    skipped = 0
    for i in range(len(imgs)):
        cleaned = remove_hair_adaptive(imgs[i])
        # Check if the image was returned unchanged
        if np.array_equal(imgs[i], cleaned):
            skipped += 1
        out[i] = cleaned
    print(f"Processed {len(imgs)} images. Skipped {skipped} images (no hair detected).")
    return out

# Run the improved version
train_images_clean = process_batch_adaptive(train_images)
test_images_clean = process_batch_adaptive(test_images)


In [ ]:
import matplotlib.pyplot as plt
import random

def visualize_classes_comparison(images_orig, images_clean, labels, idx_to_class, n_samples=3):
    # Get unique classes from the labels
    unique_classes = np.unique(labels)
    n_classes = len(unique_classes)
    
    # Create a large figure: rows = classes, cols = samples (each sample is a pair)
    # We use sub-grids to keep the Original/Cleaned pairs together
    fig, axes = plt.subplots(n_classes, n_samples, figsize=(n_samples * 3, n_classes * 3))
    
    for row, cls_idx in enumerate(unique_classes):
        class_name = idx_to_class[cls_idx]
        
        # Find all indices belonging to this class
        indices = np.where(labels == cls_idx)[0]
        
        # Pick random samples
        selected_indices = random.sample(list(indices), min(n_samples, len(indices)))
        
        for col, img_idx in enumerate(selected_indices):
            # Create a small "inner" comparison: Top half Original, Bottom half Cleaned
            # Or just stack them horizontally
            orig = images_orig[img_idx]
            clean = images_clean[img_idx]
            
            # Combine them into one image for easy grid viewing (Horizontal concatenation)
            combined = np.hstack((orig, clean))
            
            ax = axes[row, col]
            ax.imshow(combined)
            ax.axis('off')
            
            # Only set class name on the first column of each row
            if col == 0:
                ax.set_ylabel(class_name, rotation=0, size='large', labelpad=60, fontweight='bold')
                # Make the ylabel visible
                ax.axis('on')
                ax.set_xticks([])
                ax.set_yticks([])
                for spine in ax.spines.values(): spine.set_visible(False)

    plt.suptitle("Class Samples: Original (Left) vs Cleaned (Right)", fontsize=20, y=1.02)
    plt.tight_layout()
    plt.show()

print(train_images_clean.shape)
# Run the visualization
visualize_classes_comparison(train_images, train_images_clean, train_labels, idx_to_class, n_samples=3)


---
## 4. Removing reddish background

In [ ]:
# ─────────────────────────────────────────────
# STEP 2 — RED-CHANNEL CORRECTION
# Many images have a reddish background (skin + camera artefact).
# Strategy:
#   1. Check a thin border ring around the image.
#   2. If the mean red value of that ring > RED_THRESHOLD (likely reddish BG),
#      subtract a fixed amount from the red channel so the lesion
#      stands out better during colour-feature extraction.
# The original arrays are replaced in-place.
# ─────────────────────────────────────────────
import numpy as np

BORDER_W       = 8      # pixels wide border ring to sample
RED_THRESHOLD  = 150    # mean red in border → "reddish background"
RED_SUBTRACT   = 50     # how much to pull down the red channel

def has_red_background(img, border=BORDER_W, threshold=RED_THRESHOLD):
    """Return True if the image border looks reddish."""
    h, w = img.shape[:2]
    top    = img[:border,  :,    0]
    bottom = img[-border:, :,    0]
    left   = img[:,  :border,    0]
    right  = img[:, -border:,    0]
    border_red = np.concatenate([top.ravel(), bottom.ravel(),
                                 left.ravel(), right.ravel()])
    return border_red.mean() > threshold

def correct_red_channel(img, amount=RED_SUBTRACT):
    """Subtract `amount` from the red channel, clip to [0, 255]."""
    corrected = img.astype(np.int16)
    corrected[..., 0] = np.clip(corrected[..., 0] - amount, 0, 255)
    return corrected.astype(np.uint8)

def apply_red_correction(images):
    result = images.copy()
    n_corrected = 0
    for i in range(len(result)):
        if has_red_background(result[i]):
            result[i] = correct_red_channel(result[i])
            n_corrected += 1
    print(f"  Corrected {n_corrected}/{len(result)} images ({n_corrected/len(result)*100:.1f}%)")
    return result

print("Train images:")
train_images = apply_red_correction(train_images_clean)

print("Test images:")
test_images  = apply_red_correction(test_images_clean)

# ── Quick visual sanity-check (3 samples per class) ──
import matplotlib.pyplot as plt

class_names = metadata['class_names']
n_samples_per_class = 3
fig, axes = plt.subplots(2*n_samples_per_class, len(class_names), figsize=(18, 12))

for col, cls_name in enumerate(class_names):
    cls_idx  = metadata['class_to_idx'][cls_name]
    pos      = np.where(train_labels == cls_idx)[0]
    if len(pos) == 0:
        continue
    
    for row_sample in range(n_samples_per_class):
        idx_in_pos = min(row_sample, len(pos) - 1)  # use available samples
        orig_img = train_images_clean[pos[idx_in_pos]]          # before correction
        corr_img = train_images[pos[idx_in_pos]]              # after correction

        # Original row
        axes[2*row_sample, col].imshow(orig_img)
        if row_sample == 0:
            axes[2*row_sample, col].set_title(cls_name, fontsize=9)
        axes[2*row_sample, col].axis('off')

        # Corrected row
        axes[2*row_sample+1, col].imshow(corr_img)
        axes[2*row_sample+1, col].axis('off')

# Label the row groups
for row_sample in range(n_samples_per_class):
    axes[2*row_sample, 0].set_ylabel(f"Original {row_sample+1}", fontsize=8)
    axes[2*row_sample+1, 0].set_ylabel(f"Corrected {row_sample+1}", fontsize=8)

plt.suptitle(f"Red-channel correction — {n_samples_per_class} samples per class", fontsize=11)
plt.tight_layout()
plt.show()
print("Red-channel correction done ✓")


---
## 5. Correcting imbalance ('nv' undersampling)

In [ ]:
# ─────────────────────────────────────────────
# UNDERSAMPLE 'nv' (class 4) 
# Must be done BEFORE the train/val/test split
# ─────────────────────────────────────────────

nv_class = metadata['class_to_idx']['nv']   # → 4

# Indices for nv and everything else
idx_nv    = np.where(train_labels == nv_class)[0]
idx_other = np.where(train_labels != nv_class)[0]

print(f"Before undersampling:")
print(f"  nv samples   : {len(idx_nv)}")
print(f"  other samples: {len(idx_other)}")
print(f"  total        : {len(train_labels)}")

# Keep a random 50% of nv
np.random.seed(42)
idx_nv_keep = np.random.choice(idx_nv, size=len(idx_nv) // 4, replace=False)

# Combine and sort to preserve original ordering
idx_keep = np.sort(np.concatenate([idx_other, idx_nv_keep]))

# Apply to all arrays simultaneously so nothing goes out of sync
train_images  = train_images[idx_keep]
train_tabular = train_tabular[idx_keep]
train_labels  = train_labels[idx_keep]
train_ids     = train_ids[idx_keep]

# Metadata for class names
metadata = np.load(os.path.join(DATA_DIR, "metadata.npy"), allow_pickle=True).item()
print("\nMetadata content:")
print("Class names:", metadata['class_names'])

print(f"\nAfter undersampling:")
for i, name in enumerate(metadata['class_names']):
    count = (train_labels == i).sum()
    bar   = '█' * (count // 10)
    print(f"  {name:<6}  {count:>4}  {bar}")
print(f"  total        : {len(train_labels)}")

---
## 6. Split data to (Train/Validation/test)

In [ ]:
from sklearn.model_selection import train_test_split

# ─────────────────────────────────────────────
# TRAIN / VAL / TEST SPLIT  (from labeled data only)
# The 1000 test_images have no labels → used only for final submission
# ─────────────────────────────────────────────

# Indices for stratified split — keeps class proportions identical in all splits
indices = np.arange(len(train_images))

# 70% train | 15% val | 15% test
idx_train, idx_temp = train_test_split(
    indices,
    test_size=0.30,
    stratify=train_labels,
    random_state=42
)
idx_val, idx_test = train_test_split(
    idx_temp,
    test_size=0.50,          # 0.50 × 0.30 = 0.15 of total
    stratify=train_labels[idx_temp],
    random_state=42
)

# ── Image splits ──
X_img_train = train_images[idx_train]
X_img_val   = train_images[idx_val]
X_img_test  = train_images[idx_test]

# ── Tabular splits ──
X_tab_train = train_tabular[idx_train]
X_tab_val   = train_tabular[idx_val]
X_tab_test  = train_tabular[idx_test]

# ── Label splits ──
y_train = train_labels[idx_train]
y_val   = train_labels[idx_val]
y_test  = train_labels[idx_test]

# ── ID splits (useful for debugging) ──
ids_train = train_ids[idx_train]
ids_val   = train_ids[idx_val]
ids_test  = train_ids[idx_test]

print("Split sizes")
print(f"  Train : {len(idx_train):>4}  ({len(idx_train)/len(indices)*100:.0f}%)")
print(f"  Val   : {len(idx_val):>4}  ({len(idx_val)/len(indices)*100:.0f}%)")
print(f"  Test  : {len(idx_test):>4}  ({len(idx_test)/len(indices)*100:.0f}%)")
print(f"  Submission (unlabeled): {len(test_images)}")

# Verify class balance is preserved
print("\nClass distribution per split:")
class_names = metadata['class_names']
header = f"{'Class':<8}" + f"{'Train':>8}" + f"{'Val':>8}" + f"{'Test':>8}"
print(header)
for i, name in enumerate(class_names):
    tr = (y_train == i).sum()
    va = (y_val   == i).sum()
    te = (y_test  == i).sum()
    print(f"  {name:<6}  {tr:>6}  {va:>6}  {te:>6}")

---
## 7. Data Augmentation

In [ ]:
# ─────────────────────────────────────────────
# STEP 3 — DATA AUGMENTATION (balance minority classes)
# Apply ONLY to the train split to avoid data leakage
# Target: oversample every class except 'nv' up to
#         the median count of non-nv classes (rounded up),
#         keeping tabular data and IDs in sync.
# Augmentation ops: horizontal flip, vertical flip,
#                   90/180/270° rotations, brightness jitter.
# ─────────────────────────────────────────────
import numpy as np
import cv2
import matplotlib.pyplot as plt
import random

nv_class   = metadata['class_to_idx']['nv']
class_names = metadata['class_names']

# ── counts before augmentation ──
unique, counts = np.unique(y_train, return_counts=True)
count_dict = dict(zip(unique.tolist(), counts.tolist()))
print("Class counts BEFORE augmentation (train split only):")
for i, name in enumerate(class_names):
    bar = '█' * (count_dict.get(i, 0) // 5)
    print(f"  {name:<6}  {count_dict.get(i,0):>5}  {bar}")

# ── target: median count of non-nv classes ──
non_nv_counts = [count_dict[i] for i in unique if i != nv_class]
target_count  = int(np.median(non_nv_counts)) * 2 
print(f"\nAugmentation target per class (non-nv): {target_count}")

# ── augmentation functions ──
def aug_flip_h(img):  return cv2.flip(img, 1)
def aug_flip_v(img):  return cv2.flip(img, 0)
def aug_rot90(img):   return cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
def aug_rot180(img):  return cv2.rotate(img, cv2.ROTATE_180)
def aug_rot270(img):  return cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)
def aug_brightness(img, delta=25):
    out = img.astype(np.int16) + np.random.randint(-delta, delta)
    return np.clip(out, 0, 255).astype(np.uint8)

AUG_FUNCS = [aug_flip_h, aug_flip_v, aug_rot90, aug_rot180, aug_rot270, aug_brightness]

def augment_class(images, tabular, labels, ids, cls_idx, target):
    """Generate augmented samples for one class until `target` is reached."""
    src_mask  = labels == cls_idx
    src_imgs  = images[src_mask]
    src_tab   = tabular[src_mask]
    src_ids   = ids[src_mask]
    n_have    = len(src_imgs)
    n_need    = target - n_have
    if n_need <= 0:
        return np.empty((0,)+images.shape[1:], dtype=np.uint8), \
               np.empty((0, tabular.shape[1])), \
               np.array([], dtype=labels.dtype), \
               np.array([], dtype=ids.dtype)

    new_imgs  = []
    new_tab   = []
    new_ids   = []

    for k in range(n_need):
        idx       = k % n_have
        base_img  = src_imgs[idx]
        aug_fn    = random.choice(AUG_FUNCS)
        new_imgs.append(aug_fn(base_img))
        new_tab.append(src_tab[idx])           # tabular features stay identical
        new_ids.append(src_ids[idx] + f"_aug{k:04d}")   # unique augmented ID

    return (np.array(new_imgs, dtype=np.uint8),
            np.array(new_tab),
            np.full(n_need, cls_idx, dtype=labels.dtype),
            np.array(new_ids, dtype=ids.dtype))

# ── run augmentation for the TRAIN SPLIT only ──
aug_imgs_list = [X_img_train]
aug_tab_list  = [X_tab_train]
aug_lbl_list  = [y_train]
aug_id_list   = [ids_train]

for cls_idx in range(len(class_names)):
    if cls_idx == nv_class:
        continue
    ai, at, al, aid = augment_class(
        X_img_train, X_tab_train, y_train, ids_train,
        cls_idx, target_count
    )
    if len(ai):
        aug_imgs_list.append(ai)
        aug_tab_list.append(at)
        aug_lbl_list.append(al)
        aug_id_list.append(aid)

# ── merge & shuffle ──
X_img_train_aug  = np.concatenate(aug_imgs_list, axis=0)
X_tab_train_aug  = np.concatenate(aug_tab_list,  axis=0)
y_train_aug      = np.concatenate(aug_lbl_list,  axis=0)
ids_train_aug    = np.concatenate(aug_id_list,   axis=0)

shuffle_idx = np.random.permutation(len(X_img_train_aug))
X_img_train  = X_img_train_aug[shuffle_idx]
X_tab_train  = X_tab_train_aug[shuffle_idx]
y_train      = y_train_aug[shuffle_idx]
ids_train    = ids_train_aug[shuffle_idx]

# ── counts after augmentation ──
unique2, counts2 = np.unique(y_train, return_counts=True)
count_dict2 = dict(zip(unique2.tolist(), counts2.tolist()))
print("\nClass counts AFTER augmentation (train split only):")
for i, name in enumerate(class_names):
    c   = count_dict2.get(i, 0)
    bar = '█' * (c // 20)
    print(f"  {name:<6}  {c:>5}  {bar}")
print(f"  total   {len(y_train):>6}")

# ── Bar chart comparison ──
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
x = np.arange(len(class_names))
axes[0].bar(x, [count_dict.get(i,0) for i in range(len(class_names))],
            color='steelblue')
axes[0].set_xticks(x); axes[0].set_xticklabels(class_names, rotation=45)
axes[0].set_title("Before augmentation"); axes[0].set_ylabel("Count")

axes[1].bar(x, [count_dict2.get(i,0) for i in range(len(class_names))],
            color='darkorange')
axes[1].set_xticks(x); axes[1].set_xticklabels(class_names, rotation=45)
axes[1].set_title(f"After augmentation (target={target_count} per non-nv class)")

plt.tight_layout()
plt.show()
print("\n✓ Augmentation done on TRAIN SPLIT only (no data leakage) ✓  |  Tabular & IDs kept in sync ✓")


---
## 8. Segmentation

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def segment_lesion_grabcut(image):
    img_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    h, w = img_bgr.shape[:2]
    
    # --- Preprocessing ---
    # 1. Enhance contrast with CLAHE on LAB
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    enhanced = cv2.merge([l, a, b])
    enhanced = cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)
    
    # 2. Sharpen edges to help GrabCut find boundaries
    kernel_sharp = np.array([[0, -1, 0],
                              [-1, 5,-1],
                              [0, -1, 0]])
    enhanced = cv2.filter2D(enhanced, -1, kernel_sharp)
    
    # --- GrabCut ---
    margin = int(min(h, w) * 0.15)
    rect = (margin, margin, w - 2 * margin, h - 2 * margin)
    
    mask = np.zeros((h, w), np.uint8)
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)
    
    cv2.grabCut(enhanced, mask, rect, bgd_model, fgd_model, 10, cv2.GC_INIT_WITH_RECT)
    cv2.grabCut(enhanced, mask, rect, bgd_model, fgd_model, 5, cv2.GC_EVAL)
    
    final_mask = np.where((mask == 2) | (mask == 0), 0, 255).astype(np.uint8)
    
    # --- Check if mask failed ---
    foreground_ratio = final_mask.sum() / (255 * h * w)
    
    # If failed, try with Otsu thresholding as alternative
    if foreground_ratio < 0.1:
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)
        _, otsu_mask = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        final_mask = otsu_mask
    
    # --- Morphological cleanup ---
    kernel = np.ones((5, 5), np.uint8)
    final_mask = cv2.morphologyEx(final_mask, cv2.MORPH_CLOSE, kernel)
    final_mask = cv2.morphologyEx(final_mask, cv2.MORPH_OPEN, kernel)
    
    result = image.copy()
    result[final_mask == 0] = 0
    
    return final_mask, result
    


In [ ]:
import os
print(X_img_train.shape)

def process_segmentation_batch(images, split_name=""):
    """
    Applies segment_lesion_grabcut to a batch of images.
    Returns: (segmented_images, binary_masks)
    """
    n = len(images)
    segmented_out = np.empty_like(images)
    masks_out = np.empty((n, images.shape[1], images.shape[2]), dtype=np.uint8)
    
    print(f"Starting segmentation on {n} images ({split_name})...")
    for i in range(n):
        mask, segmented = segment_lesion_grabcut(images[i])
        segmented_out[i] = segmented
        masks_out[i] = mask
        
        if (i + 1) % 500 == 0:
            print(f"  Processed {i + 1}/{n} images...")
            
    return segmented_out, masks_out

# ─────────────────────────────────────────────
# APPLY MASKING TO THE SPLIT DATA
# This uses the post-augmentation, post-undersampling split arrays.
# ─────────────────────────────────────────────
print("Applying segmentation to split images...\n")

X_img_train_seg, X_masks_train = process_segmentation_batch(X_img_train, "train split")
X_img_val_seg,   X_masks_val   = process_segmentation_batch(X_img_val,   "val split")
X_img_test_seg,  X_masks_test  = process_segmentation_batch(X_img_test,  "test split")

# Keep the unlabeled submission set separate for final app/submission use.
test_images_seg, test_masks = process_segmentation_batch(test_images, "unlabeled submission set")

print("\n✓ Segmentation complete for all splits")
print(f"  Train: {X_img_train_seg.shape}  |  Masks: {X_masks_train.shape}")
print(f"  Val  : {X_img_val_seg.shape}  |  Masks: {X_masks_val.shape}")
print(f"  Test : {X_img_test_seg.shape}  |  Masks: {X_masks_test.shape}")
print(f"  Submission: {test_images_seg.shape}  |  Masks: {test_masks.shape}")


---
## 9. Saving the final processed data

In [ ]:
# ─────────────────────────────────────────────
# SAVE PROCESSED DATA AS .NPY SPLITS
# Exports train/val/test folders with SEGMENTED images, tabular data, labels, and ids.
# Will NOT save masks. Also writes PNG copies into a `png/` folder inside each split.
# ─────────────────────────────────────────────

import os
import numpy as np

# Place output next to the project root to match existing exports
OUTPUT_DIR = os.path.join(os.path.dirname(DATA_DIR), "processed_data_seg_npy")
os.makedirs(OUTPUT_DIR, exist_ok=True)

splits = {
    "train": {
        "images": X_img_train_seg,
        "tabular": X_tab_train,
        "labels": y_train,
        "ids": ids_train,
    },
    "val": {
        "images": X_img_val_seg,
        "tabular": X_tab_val,
        "labels": y_val,
        "ids": ids_val,
    },
    "test": {
        "images": X_img_test_seg,
        "tabular": X_tab_test,
        "labels": y_test,
        "ids": ids_test,
    },
}

for split_name, split_data in splits.items():
    split_dir = os.path.join(OUTPUT_DIR, split_name)
    png_dir = os.path.join(split_dir, "png")
    os.makedirs(split_dir, exist_ok=True)
    os.makedirs(png_dir, exist_ok=True)

    # Save arrays (images, tabular, labels, ids). DO NOT save masks here.
    np.save(os.path.join(split_dir, f"{split_name}_images.npy"), split_data["images"])
    np.save(os.path.join(split_dir, f"{split_name}_tabular.npy"), split_data["tabular"])
    np.save(os.path.join(split_dir, f"{split_name}_labels.npy"), split_data["labels"])
    np.save(os.path.join(split_dir, f"{split_name}_ids.npy"), split_data["ids"])

    # Also write PNG copies for quick inspection (one file per image)
    import cv2
    for i, img in enumerate(split_data['images']):
        png_path = os.path.join(png_dir, f"{split_name}_{i:05d}.png")
        cv2.imwrite(png_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

print(f"Saved processed splits to: {OUTPUT_DIR}")
for split_name in splits:
    split_dir = os.path.join(OUTPUT_DIR, split_name)
    print(f"  {split_name}: {os.listdir(split_dir)}")


In [ ]:
# ─────────────────────────────────────────────
# VERIFY SAVED .NPY FILES AS PNG PREVIEWS
# Loads the saved segmented image arrays and writes a few PNGs for inspection.
# ─────────────────────────────────────────────

import os
import numpy as np
import matplotlib.pyplot as plt
import cv2


PREVIEW_DIR = os.path.join(OUTPUT_DIR, "preview_png")
os.makedirs(PREVIEW_DIR, exist_ok=True)

preview_splits = ["train", "val", "test"]
preview_count = 3

fig, axes = plt.subplots(len(preview_splits), preview_count, figsize=(preview_count * 4, len(preview_splits) * 4))
if len(preview_splits) == 1:
    axes = np.expand_dims(axes, axis=0)

for row, split_name in enumerate(preview_splits):
    split_dir = os.path.join(OUTPUT_DIR, split_name)
    images = np.load(os.path.join(split_dir, f"{split_name}_images.npy"))

    for col in range(preview_count):
        img_idx = min(col, len(images) - 1)
        img = images[img_idx]

        png_path = os.path.join(PREVIEW_DIR, f"{split_name}_{img_idx:03d}.png")
        cv2.imwrite(png_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

        ax = axes[row, col]
        ax.imshow(img)
        ax.set_title(f"{split_name} #{img_idx}")
        ax.axis("off")

plt.tight_layout()
plt.show()

print(f"Saved PNG previews to: {PREVIEW_DIR}")
